# Part 3: Scalable Processing — Polars Query Optimization, Pandas Benchmark & PySpark Parity
Examines Polars lazy query optimization (predicate pushdown, projection pushdown, parallel window execution), benchmarks execution time and peak RSS against Pandas (naive vs column-pushdown), and verifies exact parity with PySpark.

In [1]:
from pathlib import Path
import polars as pl

print('=== Polars Optimized Query Plan ===')
print(Path('../outputs/tables/polars_optimized_plan.txt').read_text())

=== Polars Optimized Query Plan ===
=== UNOPTIMIZED LOGICAL PLAN ===
 WITH_COLUMNS:
 [when(col("qidx").shift([dyn int: 4]).over([col("CERT")]) == (col("qidx") - 4)).then((col("ASSET") / col("ASSET").shift([dyn int: 4]).over([col("CERT")])) - 1.0).otherwise(null.cast(Float64)).alias("asset_growth_4q"), when(col("qidx").shift([dyn int: 4]).over([col("CERT")]) == (col("qidx") - 4)).then(col("texas_ratio") - col("texas_ratio").shift([dyn int: 4]).over([col("CERT")])).otherwise(null.cast(Float64)).alias("d4_texas_ratio")] 
   WITH_COLUMNS:
   [when(col("quarter") == 1).then(col("NETINC")).otherwise(when(col("qidx").shift([dyn int: 1]).over([col("CERT"), col("year")]) == (col("qidx") - 1)).then(col("NETINC") - col("NETINC").shift([dyn int: 1]).over([col("CERT"), col("year")])).otherwise(null.cast(Float64))).alias("NETINC_Q"), (col("EQ") / col("ASSET")).alias("eq_asset_ratio"), when((col("EQ").fill_null([0.0]) + col("LNATRES").fill_null([0.0])) > 0.0).then((((col("NCLNLS").fill_null([0.0]) + 

In [2]:
bench = pl.read_csv('../outputs/tables/part3_benchmark_results.csv')
print('=== Part 3 Scaling & Memory Benchmark ===')
print(bench)

=== Part 3 Scaling & Memory Benchmark ===
shape: (15, 6)
┌──────────────────┬─────────┬───────────────────┬────────┬─────────────┬──────────────────────────┐
│ slice            ┆ rows    ┆ engine            ┆ time_s ┆ peak_rss_mb ┆ max_rel_diff_vs_polars_l │
│ ---              ┆ ---     ┆ ---               ┆ ---    ┆ ---         ┆ azy                      │
│ str              ┆ i64     ┆ str               ┆ f64    ┆ f64         ┆ ---                      │
│                  ┆         ┆                   ┆        ┆             ┆ f64                      │
╞══════════════════╪═════════╪═══════════════════╪════════╪═════════════╪══════════════════════════╡
│ 2010-2026        ┆ 386056  ┆ Polars lazy       ┆ 0.318  ┆ 421.0       ┆ 0.0                      │
│ 2010-2026        ┆ 386056  ┆ Polars streaming  ┆ 0.151  ┆ 348.0       ┆ 2.3000e-16               │
│ 2010-2026        ┆ 386056  ┆ Polars eager      ┆ 0.367  ┆ 640.0       ┆ 0.0                      │
│ 2010-2026        ┆ 386056  ┆ pan

In [3]:
import json
parity = json.loads(Path('../outputs/tables/spark_polars_parity.json').read_text())
print('=== PySpark vs Polars Parity Check ===')
print(json.dumps(parity, indent=2))
print('\n=== PySpark Catalyst Execution Plan (Excerpt) ===')
print(Path('../outputs/tables/spark_execution_plan.txt').read_text()[:1500])

=== PySpark vs Polars Parity Check ===
{
  "yearly_fingerprint": {
    "years_compared": 43,
    "row_and_null_counts_identical": true,
    "max_relative_diff_of_sums": 2.2382624412157772e-15
  },
  "row_level_sample": {
    "banks_sampled": 483,
    "rows_compared": 33042,
    "rows_only_in_one_engine": 0,
    "max_abs_diff_any_column": 0.0
  },
  "timing_seconds": {
    "polars_lazy_collect_plus_aggregate": 0.62,
    "spark_session_startup": 3.98,
    "spark_job_cold": 4.97,
    "spark_job_warm": 2.18,
    "note": "single in-process runs on the full panel; Spark = local[2], 1.2 GB driver"
  }
}

=== PySpark Catalyst Execution Plan (Excerpt) ===
== Physical Plan ==
AdaptiveSparkPlan (12)
+- Project (11)
   +- Window (10)
      +- Sort (9)
         +- Exchange (8)
            +- Project (7)
               +- Window (6)
                  +- Sort (5)
                     +- Exchange (4)
                        +- Project (3)
                           +- Project (2)
                     